# Task 2 — Sentiment Analysis

This notebook runs sentiment analysis on the cleaned review dataset using DistilBERT.

**Model:** `distilbert-base-uncased-finetuned-sst-2-english`  
**Input:** `data/processed/reviews_clean.csv`  
**Output:** `data/processed/reviews_with_sentiment.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.sentiment_analysis import load_data, init_model, run_sentiment, print_summary, save_output
from src.config import PathConfig, SentimentConfig

paths = PathConfig()
sentiment_cfg = SentimentConfig()

print('Config loaded')
print(f'  Model: {sentiment_cfg.model_name}')
print(f'  Threshold: {sentiment_cfg.threshold}')
print(f'  Input: {paths.processed_reviews}')

In [ ]:
# Load cleaned data
df = load_data(paths.processed_reviews)
df.head()

In [ ]:
# Basic stats before sentiment
print(f'Total reviews: {len(df):,}')
print(f'\nReviews per bank:')
print(df['bank'].value_counts().to_string())
print(f'\nAverage rating per bank:')
print(df.groupby('bank')['rating'].mean().round(2).to_string())

In [ ]:
# Run sentiment analysis
# Note: first run downloads the DistilBERT model (~250MB)
model = init_model(sentiment_cfg)
df = run_sentiment(df, model, batch_size=sentiment_cfg.batch_size)
print('Sentiment analysis complete')
df[['bank', 'review', 'rating', 'sentiment_label', 'sentiment_score']].head(10)

In [ ]:
# Summary statistics
print_summary(df)

In [ ]:
# Visualise sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall sentiment
sentiment_counts = df['sentiment_label'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(sentiment_counts.values, labels=sentiment_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Overall Sentiment Distribution', fontsize=13, fontweight='bold')

# By bank
bank_sentiment = df.groupby(['bank', 'sentiment_label']).size().unstack(fill_value=0)
bank_sentiment.plot(kind='bar', ax=axes[1], color=colors)
axes[1].set_title('Sentiment by Bank', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Bank')
axes[1].set_ylabel('Number of Reviews')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(['Negative', 'Positive'])

plt.tight_layout()
plt.savefig('../reports/figures/sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to reports/figures/sentiment_distribution.png')

In [ ]:
# Confidence score distribution
plt.figure(figsize=(10, 4))
for bank in df['bank'].unique():
    subset = df[df['bank'] == bank]
    plt.hist(subset['sentiment_score'], bins=20, alpha=0.6, label=bank)
plt.xlabel('Sentiment Confidence Score')
plt.ylabel('Count')
plt.title('Sentiment Confidence Score Distribution by Bank')
plt.legend()
plt.axvline(x=0.7, color='red', linestyle='--', label='Threshold (0.7)')
plt.tight_layout()
plt.show()
print(f'Reviews below threshold (0.7): {(df["sentiment_score"] < 0.7).sum()}')

In [ ]:
# Save output
save_output(df, paths.sentiment_output)
print(f'\nSaved {len(df):,} reviews with sentiment to {paths.sentiment_output}')